# 03 · DPO (Direct Preference Optimization) 原理与实战

> 前置:先跑通 [`00`](00_环境准备与GPU检测.ipynb)。本章会用 **4-bit 底座 + LoRA** 真训练,单张 16GB 可跑。
> 这一章**不依赖** 02 的奖励模型 —— 这正是 DPO 的最大卖点。

**DPO 是目前最推荐的入门对齐方法**:它**不需要**单独训练奖励模型,也**不需要**在线采样,
直接拿一批 `(问题, 好回答 chosen, 坏回答 rejected)` 数据,用一个巧妙的损失函数,一步到位优化策略。
又稳、又省资源,和 LoRA/QLoRA 是天生一对。

## 一、原理:DPO 是怎么「跳过奖励模型」的

回忆 RLHF 的目标:在**最大化奖励**的同时,用 KL 惩罚**别偏离参考模型太远**。
DPO 的作者做了一个漂亮的数学推导:这个带 KL 约束的最优策略,其实存在**闭式解**,
把奖励 \(r\) 反解出来后代入 Bradley-Terry,奖励模型就被「消掉」了,最终只剩下一个可以直接对策略求导的损失:

\[ \mathcal{L}_{DPO} = -\,\mathbb{E}\Big[\log \sigma\Big(\beta \log \tfrac{\pi_\theta(y_c\mid x)}{\pi_{ref}(y_c\mid x)} - \beta \log \tfrac{\pi_\theta(y_r\mid x)}{\pi_{ref}(y_r\mid x)}\Big)\Big] \]

拆开看其实很直观:

- \(\log \frac{\pi_\theta(y)}{\pi_{ref}(y)}\) 就是 DPO 里的**隐式奖励**(相对参考模型,策略对这段回答「加了多少分」);
- 损失让 **chosen 的隐式奖励 > rejected 的隐式奖励**;
- \(\beta\) 控制「偏离参考模型的强度」:\(\beta\) 越大,越保守(越贴近参考模型);越小,越敢改。

**和奖励模型(02)对比:** RM 训练的是「一个外部打分器」;DPO 直接把这个思想「内化」进策略本身的对数概率比里。

## 二、加载偏好数据(优先用 HuggingFace 真实数据)

和 02 一样,**优先从 HuggingFace 拉取** [`llamafactory/DPO-En-Zh-20k`](https://huggingface.co/datasets/llamafactory/DPO-En-Zh-20k)(`zh` 子集),失败自动回退本地 `data/preference.jsonl`。

DPO 数据需要三列:`prompt` / `chosen` / `rejected`。因为底座是 **Instruct 对话模型**,我们用**对话格式**(显式 prompt),让 `DPOTrainer` 自动套 chat template:

- `prompt` = 用户提问(历史对话);
- `chosen` = 助手的**更优**回答;`rejected` = 助手的**更差**回答。

> 演示只取一小部分(`N_SAMPLES`),想要明显效果就调大它并增大 `max_steps`。

In [ ]:
import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
from datasets import load_dataset

N_SAMPLES = 200                                        # 演示规模:偏好数据只取一小部分
_ROLE = {"human": "user", "gpt": "assistant", "system": "system"}

def load_preference_for_dpo():
    """优先用 HuggingFace 中文偏好数据 llamafactory/DPO-En-Zh-20k(zh);失败回退本地 jsonl。
    统一整理成 DPOTrainer 需要的『显式 prompt 对话格式』:prompt / chosen / rejected。"""
    try:
        ds = load_dataset("llamafactory/DPO-En-Zh-20k", "zh", split="train")
        def conv(ex):
            prompt = [{"role": _ROLE.get(m["from"], "user"), "content": m["value"]}
                      for m in ex["conversations"]]
            return {
                "prompt":   prompt,
                "chosen":   [{"role": "assistant", "content": ex["chosen"]["value"]}],
                "rejected": [{"role": "assistant", "content": ex["rejected"]["value"]}],
            }
        ds = ds.map(conv, remove_columns=ds.column_names)
        ds = ds.shuffle(seed=42).select(range(min(N_SAMPLES, len(ds))))
        print(f"✅ 已从 HuggingFace 加载 llamafactory/DPO-En-Zh-20k (zh),取 {len(ds)} 条")
        return ds
    except Exception as e:
        print(f"⚠️ 从 HF 下载失败({type(e).__name__}),回退本地 data/preference.jsonl\n   {e}")
        raw = load_dataset("json", data_files="../data/preference.jsonl", split="train")
        def conv(ex):
            return {
                "prompt":   [{"role": "user", "content": ex["prompt"]}],
                "chosen":   [{"role": "assistant", "content": ex["chosen"]}],
                "rejected": [{"role": "assistant", "content": ex["rejected"]}],
            }
        return raw.map(conv, remove_columns=raw.column_names)

dpo_ds = load_preference_for_dpo()
print(dpo_ds)
print(dpo_ds[0])

## 三、用 4-bit 加载策略模型(QLoRA 思路)

- 底座用 `Qwen2.5-1.5B-Instruct`,**4-bit NF4** 量化加载,大幅省显存。
- `prepare_model_for_kbit_training` 做量化训练前的准备。
- **参考模型 \(\pi_{ref}\) 不用单独再加载一份**:用 LoRA 时,`ref_model=None`,`DPOTrainer` 会在计算参考项时**临时关闭 LoRA 适配器**,此时模型就等价于原始底座 —— 省下一整个模型的显存。

> 承接 test-lora:如果你完成了那边的 SFT,可把 `MODEL_NAME` 改成 test-lora 合并好的模型目录,
> 让「SFT 模型」真正成为「策略模型的起点」。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
# 承接 test-lora 的 SFT 模型(可选):
# MODEL_NAME = "../../test-lora/outputs/merged-qwen1.5b"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    torch_dtype=torch.bfloat16, device_map="cuda",
)
model = prepare_model_for_kbit_training(model)
print("策略模型(4-bit)就绪。参考模型将由『关闭 LoRA 适配器』充当(ref_model=None)。")

## 四、配置 LoRA + DPOConfig,开始训练

- `beta`:DPO 的核心超参,控制偏离参考模型的强度(常用 0.1)。
- `max_length` / `max_prompt_length`:控制序列长度,越小越省显存。
- 一次前向要同时过 chosen 和 rejected,所以 batch 设得比 SFT 更小。

In [ ]:
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

dpo_args = DPOConfig(
    output_dir="../outputs/dpo-qwen1.5b",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    beta=0.1,                 # 偏离参考模型的强度
    max_steps=40,             # 玩具规模;想要明显效果调大(如 200+)
    max_length=512,
    max_prompt_length=256,
    logging_steps=5,
    report_to=[],
    bf16=True,
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,           # 用 LoRA 时留空:靠临时禁用适配器当参考
    args=dpo_args,
    train_dataset=dpo_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.train()
trainer.save_model("../outputs/dpo-qwen1.5b")
print("DPO 适配器已保存到 ../outputs/dpo-qwen1.5b(06 评估会用到)。")

## 五、对齐前后对比

同一批问题,分别看**对齐前**(`disable_adapter()` 关掉 LoRA,等价原始底座)和**对齐后**的回答。
重点看:对齐后是否更礼貌、更贴合我们偏好数据的风格(比如带上「—— 你的学习小助手」),以及对「作弊」这类请求是否更倾向拒绝。

> 只跑了 40 步的话,差异可能只是「初见端倪」。想效果明显,请调大 `max_steps` 并补充数据后重训。

In [ ]:
import torch

policy = trainer.model
policy.eval()

@torch.no_grad()
def gen(question, max_new_tokens=96):
    msgs = [{"role": "user", "content": question}]
    ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(policy.device)
    out = policy.generate(ids, max_new_tokens=max_new_tokens, do_sample=False,
                          pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

for q in ["什么是过拟合?", "帮我把这句话改得更礼貌一些:把报告发给我。", "教我怎么在考试里作弊。"]:
    with policy.disable_adapter():
        before = gen(q)
    after = gen(q)
    print("=" * 72)
    print(f"【问题】{q}")
    print(f"\n[对齐前]\n{before}")
    print(f"\n[对齐后]\n{after}")
print("=" * 72)

## 小结

- DPO 用一个巧妙的损失,把「奖励模型 + KL 约束」直接内化到策略的**对数概率比**里,**跳过了显式奖励模型和在线采样**。
- 隐式奖励 = \(\log \frac{\pi_\theta}{\pi_{ref}}\);`beta` 控制保守程度。
- 用 LoRA 时 `ref_model=None`,靠禁用适配器当参考,**只需一份模型权重**,4-bit 底座上单卡就能做。
- 稳、省、简单,是当下最推荐的入门对齐方法。

下一站:[`04_PPO原理与实战`](04_PPO原理与实战.ipynb) —— 走一遍更经典也更重的 RLHF 闭环,用上 02 的奖励模型。